In [ ]:
# =============================================================================
# COMPLETE LCA WORKFLOW 
# =============================================================================

# -----------------------------------------------------------------------------
# 1. CONFIGURATION: SELECT AGE GROUP TO ANALYSE
# -----------------------------------------------------------------------------
age_group = 'all'          # Options: 'all', 'child', 'teen'
# 'child' = age ≤ 11, 'teen' = age > 11

# -----------------------------------------------------------------------------
# 2. IMPORT ALL REQUIRED LIBRARIES AND MODULES
# -----------------------------------------------------------------------------
import pandas as pd
import numpy as np
import warnings
from pathlib import Path
from scipy.stats import chi2_contingency
import matplotlib.pyplot as plt
import seaborn as sns

# Import LCA functions from lca.py
from lca import perform_true_lca,plot_lca_results,LatentClassAnalysis,prepare_data_for_lca,calculate_imc_categories,apply_code_map_without_imputation,CODE_MAP


# Import utility functions from utils.py
from utils import build_full_dict,question_list,analisis_estadistico_clusters,analisis_interacciones_variables      

# -----------------------------------------------------------------------------
# 3. LOAD YOUR DATA
# -----------------------------------------------------------------------------
# Symptom data (must contain an 'id' column)
df_sintomas_disc = pd.read_csv('path/to/your/symptom_data.csv')   # e.g., 'sintomas.csv'

# Label mapping (columns: 'Código', 'Etiqueta')
df_etiquetas = pd.read_csv('path/to/your/labels.csv')

# Base dataframe with age (must contain 'inum' and 'ed1')
df_base = pd.read_csv('path/to/your/base_data.csv')               # columns: 'inum', 'ed1'

print(f"Original symptom data shape: {df_sintomas_disc.shape}")
print(f"Label mapping shape: {df_etiquetas.shape}")
print(f"DB shape: {df_base.shape}")

# -----------------------------------------------------------------------------
# 4. MERGE AGE INFORMATION AND FILTER BY AGE GROUP
# -----------------------------------------------------------------------------
# Merge age onto the symptom data using the identifier (assume 'id' in df_sintomas_disc
# corresponds to 'inum' in df_base)
df_merged = pd.merge(
    df_sintomas_disc,
    df_base[['inum', 'ed1']],
    left_on='id',
    right_on='inum',
    how='left'
).drop(columns=['inum'])

print(f"Data after merging age: {df_merged.shape}")

# Apply age filter
if age_group == 'child':
    df_work = df_merged[df_merged['ed1'] <= 11].copy()
    suffix = "_child"
elif age_group == 'teen':
    df_work = df_merged[df_merged['ed1'] > 11].copy()
    suffix = "_teen"
else:
    df_work = df_merged.copy()
    suffix = ""

print(f"\nWorking with {age_group.upper()} group: {df_work.shape[0]} observations")
print(f"Age range: {df_work['ed1'].min()} – {df_work['ed1'].max()}")

# -----------------------------------------------------------------------------
# 5. BUILD FULL DICTIONARY (optional)
# -----------------------------------------------------------------------------
# This step may use all original data; you can comment out if not needed.
full_dict = build_full_dict(df_work)
print("Full dictionary built successfully.")

# -----------------------------------------------------------------------------
# 6. GET THE LIST OF QUESTIONS FOR LCA
# -----------------------------------------------------------------------------
q_list = question_list()
print(f"Number of questions selected for LCA: {len(q_list)}")

# -----------------------------------------------------------------------------
# 7. PERFORM LCA FOR k = 2 TO 6 (EM ALGORITHM)
# -----------------------------------------------------------------------------
print("\n" + "="*70)
print(f"Running LCA with k = 2 to 6 for {age_group.upper()} group...")
print("="*70)

output_df, prob_df, metric_df = perform_true_lca(
    df=df_work,
    question_list=q_list,
    id_column='id',
    n_components_range=range(2, 7)
)

# -----------------------------------------------------------------------------
# 8. SAVE LCA OUTPUTS (with group suffix)
# -----------------------------------------------------------------------------
output_df.to_csv(f'output_df_lca{suffix}.csv', index=False)
prob_df.to_csv(f'prob_df_lca{suffix}.csv', index=False)
metric_df.to_csv(f'metric_df_lca{suffix}.csv', index=False)
print("\nLCA results saved to CSV files.")

# -----------------------------------------------------------------------------
# 9. GENERATE COMPREHENSIVE PDF REPORT (CLUSTER ANALYSIS)
# -----------------------------------------------------------------------------
print("\n" + "="*70)
print(f"Generating PDF report for {age_group.upper()} group...")
print("="*70)

resultados = analisis_estadistico_clusters(
    final_df=output_df,
    id_column='id',
    cluster_prefix='lca_k',
    df_etiquetas=df_etiquetas,
    col_codigo='Código',
    col_etiqueta='Etiqueta',
    output_pdf=f'reporte_lca{suffix}.pdf',
    metrics_df=metric_df,
    mostrar_todas_variables_perfil=True,
    max_variables_perfil=None
)

# =============================================================================
# 10. EXTRACT KEY RESULTS FROM THE 'resultados' DICTIONARY
# =============================================================================
metrics = resultados.get('metrics')
distribucion = resultados.get('distribucion')
cramers_all = resultados.get('cramers_completos')
perfiles = resultados.get('perfiles')
jerarquia = resultados.get('jerarquia')

# -----------------------------------------------------------------------------
# 11. COMPUTE PAIRWISE CRAMÉR'S V AMONG ORIGINAL VARIABLES
# -----------------------------------------------------------------------------
print("\n" + "="*70)
print("Computing pairwise Cramér's V matrices for all original variables")
print("="*70)

excluir = ['id'] + [col for col in output_df.columns if col.startswith('lca_k')]
cramersv_mat, pvalue_mat, chi2_mat = analisis_interacciones_variables(
    df=output_df,
    exclude_cols=excluir,
    max_categories=10          # adjust as needed
)

# Save matrices
cramersv_mat.to_csv(f'cramersv_pairwise_matrix{suffix}.csv')
pvalue_mat.to_csv(f'pvalue_pairwise_matrix{suffix}.csv')
chi2_mat.to_csv(f'chi2_pairwise_matrix{suffix}.csv')
print("Pairwise Cramér's V matrices saved to CSV files.")